# Drive and utilities


In [ ]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
#@title Utils
!pip install dill
!pip install pandas
!pip install numpy
!
import dill as pickle

def load(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 2.2 MB/s eta 0:00:00


# Load, clean and export

In [ ]:
#@title Load Data
!pip install numpy pandas scipy scikit-learn sklearn tqdm cudf cupy

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
from scipy import stats

## Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Load Matched Sample dictionary where keys are lambda's and values are Pandas DataFrames of matched sample
sample = load("/content/drive/MyDrive/PhD Data/11 Matches/12 Hybrid matches - top tech, filing date.pkl")

for lam, df in sample.items():
  df['treated_id'] = df['treated_id'].astype(str)
  df['control_id'] = df['control_id'].astype(str)

  print(f"Sample size : {len(df)}.")

## Get IDs in the samples
patent_ids = []

for _, df in sample.items():
  patent_ids.extend(df['treated_id'].unique())
  patent_ids.extend(df['control_id'].unique())

patent_ids = np.unique(patent_ids)

# TRIM CITATIONS DATA
# Only keep citations for patents that are either treated or control.

## Load citations as Pandas DataFrame from parquet.
citations = pd.read_pickle("/content/drive/MyDrive/PhD Data/08 Citations/03 Patent citations - raw, filing.pickle")
citations.rename(columns={'patent_id': 'citedby_patent_id', 'citation_patent_id':'patent_id', 'filing_date':'citation_date'}, inplace = True)

citations['patent_id'] = citations['patent_id'].astype(str)
citations = citations[citations['patent_id'].isin(patent_ids)]
print("Trimmed citations shape:", citations.shape)


  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Using device: cpu
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Sample size : 2035.
Trimmed citations shape: (8

In [ ]:
'93985870' in patent_ids

False

In [ ]:
#@title Precompute Treated Patent Quarterly Counts
import cudf
import pandas as pd
import numpy as np
import cudf
import pandas as pd
import numpy as np

# ================================================
# Step 1: Precompute Quarterly Citation Counts for All Patents (GPU version)
# ================================================
# Assume 'citations' is a cuDF DataFrame with columns:
#   - 'patent_id': the cited patent's ID
#   - 'citation_date': a datetime column

# Manually compute a quarter string: "YYYYQX"
citations['citation_date'] = pd.to_datetime(citations['citation_date'])
citations['year'] = citations['citation_date'].dt.year
citations['month'] = citations['citation_date'].dt.month
# Compute quarter as integer: ((month-1)//3 + 1)
citations['qtr'] = ((citations['month'] - 1) // 3 + 1).astype(str)
citations['citation_quarter'] = citations['year'].astype(str) + "Q" + citations['qtr']

# Group by patent_id and citation_quarter to count citations per quarter.
# cuDF's groupby().size() returns a Series; unstack is not supported, so convert to pandas.
grouped = citations.groupby(['patent_id', 'citation_quarter']).size()
quarterly_counts_pd = grouped.unstack(fill_value=0)

# Build a dictionary mapping patent_id -> {quarter: count}
citation_counts_dict = {}
for pid, row in quarterly_counts_pd.iterrows():
    citation_counts_dict[pid] = row.to_dict()



In [ ]:
#@title Combine with citations

# --- Step A: Augment each matched DataFrame with quarterly citation counts ---
# Define quarter labels from t-4 to t+20.
quarter_labels = [f"q_{i}" for i in range(-4, 21)]  # Labels: q_-4, q_-3, ..., q_20

for lam, df in sample.items():
    treated_cits = []  # will hold treated citation vectors (wide)
    control_cits = []  # will hold control citation vectors (wide)

    for idx, row in df.iterrows():
        # Instead of using an 'acq_quarter' column, use the stored pre-treatment quarters.
        # The treatment quarter is defined as the quarter immediately after the last pre-treatment quarter.
        pre_q_list = row['pre_quarters']  # this should be a list of quarters (e.g., ["2018Q1", "2018Q2", "2018Q3", "2018Q4"])
        treatment_period = pd.Period(pre_q_list[-1], freq='Q') + 1

        # Build a list of quarter strings from t-4 to t+20.
        quarters = ([str(treatment_period - i) for i in range(4, 0, -1)] +
                    [str(treatment_period)] +
                    [str(treatment_period + i) for i in range(1, 21)])

        treated_id = row['treated_id']
        control_id = row['control_id']
        t_vec = []
        c_vec = []
        for q in quarters:
            q_period = pd.Period(q, freq='Q')
            # For treated patent:
            if treated_id in citation_counts_dict and q in citation_counts_dict[treated_id]:
                t_vec.append(citation_counts_dict[treated_id][q])
            else:
                # If q is after treatment, assign np.nan; otherwise, assign 0.
                t_vec.append(np.nan if q_period > treatment_period else 0)
            # For control patent:
            if control_id in citation_counts_dict and q in citation_counts_dict[control_id]:
                c_vec.append(citation_counts_dict[control_id][q])
            else:
                c_vec.append(np.nan if q_period > treatment_period else 0)
        treated_cits.append(t_vec)
        control_cits.append(c_vec)

    # Add wide-format columns (one per quarter) to the DataFrame.
    for i, label in enumerate(quarter_labels):
        df[label + "_treated"] = [vec[i] for vec in treated_cits]
        df[label + "_control"] = [vec[i] for vec in control_cits]

    # Add an identifier column.
    df['match_id'] = df.index

In [ ]:
#@title Combine all and save
# --- Step B: Combine all matched DataFrames into one long-format DataFrame with lambda variable ---
dfs = []
for lam, df in sample.items():
    df = df.copy()
    df['lambda'] = lam
    # Identify columns for merging.
    id_cols = ['treated_id', 'control_id', 'match_id', 'lambda',
               'mahalanobis_distance', 'cosine_distance', 'hybrid_distance']

    # Get the wide-format citation columns.
    treated_cols = [col for col in df.columns if col.endswith("_treated") and col.startswith("q_")]
    control_cols = [col for col in df.columns if col.endswith("_control") and col.startswith("q_")]

    # Reshape to long format for treated citations.
    df_treated_long = df.melt(id_vars=id_cols,
                              value_vars=treated_cols,
                              var_name="quarter_treated",
                              value_name="citations_treated")
    # Reshape to long format for control citations.
    df_control_long = df.melt(id_vars=id_cols,
                              value_vars=control_cols,
                              var_name="quarter_control",
                              value_name="citations_control")
    # Remove suffixes to obtain a common quarter variable.
    df_treated_long['quarter'] = df_treated_long['quarter_treated'].str.replace("_treated", "")
    df_control_long['quarter'] = df_control_long['quarter_control'].str.replace("_control", "")

    # Merge treated and control long data on the id columns and quarter.
    df_long = pd.merge(df_treated_long[id_cols + ['quarter', 'citations_treated']],
                       df_control_long[id_cols + ['quarter', 'citations_control']],
                       on=id_cols + ['quarter'])
    dfs.append(df_long)

combined_df = pd.concat(dfs, ignore_index=True)
combined_df.sort_values(by=['treated_id', 'lambda', 'quarter'], inplace=True)

# --- Step C: Export the combined long-format DataFrame ---
combined_df.to_csv("/content/drive/MyDrive/PhD Data/12 Sample Final/02 Working sample - top tech, filing.csv", index=False)
print("Combined long-format DataFrame exported to drive.")


Combined long-format DataFrame exported to drive.
